In [3]:
import pandas as pd

# Load your synthetic dataset (update filename if needed)
df = pd.read_csv("synthetic_crop_dataset.csv")

# Build per-crop statistics
stats_df = df.groupby("Crop").agg({
    'Fertilizer_kg_per_ha': ['quantile','min','max','median'],
    'Pesticide_kg_per_ha': ['quantile','min','max','median'],
    'Yield_t_per_ha': ['quantile','min','max','median']
}).reset_index()

# Fix quantile extraction
def get_quantiles(series):
    return {
        'p10': series.quantile(0.10),
        'p25': series.quantile(0.25),
        'p50': series.quantile(0.50),
        'p75': series.quantile(0.75),
        'p90': series.quantile(0.90)
    }

all_stats = []
for crop, group in df.groupby("Crop"):
    fert_q = get_quantiles(group['Fertilizer_kg_per_ha'])
    pest_q = get_quantiles(group['Pesticide_kg_per_ha'])
    yield_q = get_quantiles(group['Yield_t_per_ha'])
    all_stats.append({
        'Crop': crop,
        **{f'fert_{k}': v for k,v in fert_q.items()},
        **{f'pest_{k}': v for k,v in pest_q.items()},
        **{f'yield_{k}': v for k,v in yield_q.items()}
    })

stats_df = pd.DataFrame(all_stats).set_index("Crop")

# Function for recommendation
def predict(crop: str, area_ha: float, temperature_c: float = None,
                        method: str = 'median', narrow_pct: float = 0.12) -> dict:
    crop = str(crop).lower()
    if crop not in stats_df.index.str.lower():
        raise ValueError(f"Crop '{crop}' not found. Available: {list(stats_df.index)}")

    # handle case-insensitive crop names
    matched_crop = [c for c in stats_df.index if c.lower() == crop][0]
    s = stats_df.loc[matched_crop]

    # extract percentiles
    f25, f50, f75 = float(s['fert_p25']), float(s['fert_p50']), float(s['fert_p75'])
    p25, p50, p75 = float(s['pest_p25']), float(s['pest_p50']), float(s['pest_p75'])
    y25, y50, y75 = float(s['yield_p25']), float(s['yield_p50']), float(s['yield_p75'])

    if method == 'iqr':
        fert_low_ha, fert_high_ha = f25, f75
        pest_low_ha, pest_high_ha = p25, p75
        y_low_ha, y_high_ha = y25, y75
    elif method == 'median':
        fert_low_ha, fert_high_ha = f50*(1-narrow_pct), f50*(1+narrow_pct)
        pest_low_ha, pest_high_ha = p50*(1-narrow_pct), p50*(1+narrow_pct)
        y_low_ha, y_high_ha = y50*(1-narrow_pct), y50*(1+narrow_pct)
    else:
        raise ValueError("Unknown method. Use 'iqr' or 'median'.")

    return {
        'Crop': matched_crop,
        'Area_ha': round(float(area_ha),2),
        'Temperature_C_used': None if temperature_c is None else round(float(temperature_c),1),
        'Fertilizer_per_ha_range': (round(fert_low_ha,2), round(fert_high_ha,2)),
        'Pesticide_per_ha_range': (round(pest_low_ha,3), round(pest_high_ha,3)),
        'Yield_per_ha_range': (round(y_low_ha,3), round(y_high_ha,3)),
        'Total_Fertilizer_range': (round(fert_low_ha*area_ha,2), round(fert_high_ha*area_ha,2)),
        'Total_Pesticide_range': (round(pest_low_ha*area_ha,3), round(pest_high_ha*area_ha,3)),
        'Total_Yield_range': (round(y_low_ha*area_ha,3), round(y_high_ha*area_ha,3))
    }

# # -------------------- User Input --------------------
# print("Loaded dataset. Detected crops:", list(stats_df.index))
# crop = input("Enter crop name: ").strip()
# area = float(input("Enter your farm area (in hectares): ").strip())
# temp_in = input("Enter temperature (°C) or press Enter to skip: ").strip()
# temperature = float(temp_in) if temp_in else None

# result = predict(crop, area, temperature, method='iqr')

# # -------------------- Output --------------------
# print("\n=== Recommendation (from dataset) ===")
# print(f"Crop: {result['Crop'].title()}")
# print(f"Area: {result['Area_ha']} ha")
# if result['Temperature_C_used'] is not None:
#     print(f"Temperature considered: {result['Temperature_C_used']} °C")
# print(f"Fertilizer per ha (kg): {result['Fertilizer_per_ha_range']}")
# print(f"Total Fertilizer needed (kg): {result['Total_Fertilizer_range']}")
# print(f"Pesticide per ha (kg): {result['Pesticide_per_ha_range']}")
# print(f"Total Pesticide needed (kg): {result['Total_Pesticide_range']}")
# print(f"Yield per ha (tons): {result['Yield_per_ha_range']}")
# print(f"Expected total yield (tons): {result['Total_Yield_range']}")



import pickle

# save
with open("crop_yield.pkl", "wb") as f:
    pickle.dump((stats_df, predict), f)


In [4]:
import pickle

# Specify the filename
filename = "crop_yield.pkl"

# Load the data from the pickle file
try:
    with open(filename, "rb") as f:
        stats_df_loaded, predict = pickle.load(f)

    print("Successfully loaded data from", filename)
    # You can now use stats_df_loaded and recommend_from_data_loaded
    print("\nLoaded stats_df (first 5 rows):")
    # display(stats_df_loaded.head())
    print(predict('coffee',1,25,'iqr'))

except FileNotFoundError:
    print(f"Error: The file '{filename}' was not found.")
except Exception as e:
    print(f"An error occurred while loading the file: {e}")

Successfully loaded data from crop_yield.pkl

Loaded stats_df (first 5 rows):
{'Crop': 'coffee', 'Area_ha': 1.0, 'Temperature_C_used': 25.0, 'Fertilizer_per_ha_range': (123.72, 177.5), 'Pesticide_per_ha_range': (0.566, 0.826), 'Yield_per_ha_range': (0.782, 1.49), 'Total_Fertilizer_range': (123.72, 177.5), 'Total_Pesticide_range': (0.566, 0.826), 'Total_Yield_range': (0.782, 1.49)}


This code will attempt to load the `stats_df` DataFrame and the `recommend_from_data` function from the pickle file. The `try...except` block will help identify if there's an issue with the file itself or if it simply can't be found.